# 01 — Explore region & build cloud-free composites

Study area and dates come from `src/data/regions.py`: Colniza, Mato Grosso (Brazilian arc of deforestation), with two matched dry-season windows two years apart.

> Requires `earthengine authenticate` and a Cloud project (`EE_PROJECT`).


In [ ]:
import os, sys
from pathlib import Path

# keep notebooks thin: reuse the src/ modules
sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

import ee
import geemap
from data.earth_engine import init, get_cloud_free_composite, export_bands
from data.regions import REGION, DATE1, DATE2, CLOUD_PCT

init()
print("EE initialised")

In [ ]:
region = ee.Geometry.Rectangle(REGION["bounds"])
print(REGION["name"])
print("date 1 (baseline):", DATE1)
print("date 2 (compare): ", DATE2)
region

In [ ]:
def scene_count(window):
    return (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
            .filterBounds(region).filterDate(*window)
            .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", CLOUD_PCT))
            .size().getInfo())

print("scenes 2019/2020 window:", scene_count(DATE1))
print("scenes 2021/2022 window:", scene_count(DATE2))

In [ ]:
composite1 = get_cloud_free_composite(region, *DATE1, CLOUD_PCT)
composite2 = get_cloud_free_composite(region, *DATE2, CLOUD_PCT)
composite1.bandNames().getInfo()

In [ ]:
m = geemap.Map()
m.centerObject(region, 11)
m.addLayer(composite1, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "date 1 (2020)")
m.addLayer(composite2, {"bands": ["B4", "B3", "B2"], "min": 0, "max": 3000}, "date 2 (2022)")
m.addLayer(region, {}, "AOI")
m.addLayerControl()
m

In [ ]:
# Run once to export to Google Drive for local tiling:
# export_bands(composite1, region, description="s2_colniza_2020")
# export_bands(composite2, region, description="s2_colniza_2022")